In [1]:
 !pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print('success!!!!!!!')

success!!!!!!!


In [3]:
GROP_API_KEY='------------------------------------------------------------'
os.environ['GROQ_API_KEY']=GROP_API_KEY
groq_client=Groq(api_key=GROP_API_KEY)
print('groq is initialised')

groq is initialised


In [4]:
df=pd.read_csv('college_notes.csv')
print('sahpe:',df.shape)
print('column name:',df.columns.to_list())
print('head:',df.head(3))
print('tail:',df.tail(3))

sahpe: (15, 4)
column name: ['note_id', 'subject', 'topic', 'content']
head:   note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
tail:    note_id             subject                           topic  \
12    N013       Generative AI  Retrieval Augmented Generation   
13    N014  Python Programming                  Pandas Library   
14    N015  Python Programming              Data Visualization   

                                              content  
12  RAG or Retrieval Augmented Generation is a tec...  
13  Pandas is a Python library used for data manip...  
14  Data visualization is the process of represent...  


In [5]:
print("subjects in the datasets:")
print(df['subject'].value_counts())
print("sample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of content (number of characters) of each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

subjects in the datasets:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64
sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python

In [6]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas=[
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]
print('total chunks:',len(documents))
print('first document id:',ids[0])
print('first metadata:',metadatas[0])
print('first 100 chars of doc:',documents[0][:100])

total chunks: 15
first document id: note_N001
first metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
first 100 chars of doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc


In [8]:
print('15th document id:', ids[14])
print('15th metadata:', metadatas[14])
print('first 100 chars of 15th doc:', documents[14][:100])


15th document id: note_N015
15th metadata: {'subject': 'Python Programming', 'topic': 'Data Visualization'}
first 100 chars of 15th doc: Data visualization is the process of representing data as charts graphs and visual formats. Python l


In [9]:
print('subsequent runs will be fatser')
embedding_model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('embedding model loaded')
test_embedding=embedding_model.encode('this is a test')
print('test embedding shape:',test_embedding.shape)
print('first 5 shapes:',test_embedding[:5])

subsequent runs will be fatser


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model loaded
test embedding shape: (384,)
first 5 shapes: [ 0.03061242  0.01383142 -0.02084372  0.01632787 -0.01023151]


In [10]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name='college_notes_ragg')
print('collection created')
print('documents in collection sofar:',collection.count())

collection created
documents in collection sofar: 0


In [11]:
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print('embeddings shape:',embeddings.shape)
embeddings_list=embeddings.tolist()
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list
)

print('documents succesfuly added to the list')
print('total documents in collection:',collection.count())

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (15, 384)
documents succesfuly added to the list
total documents in collection: 15


In [12]:
def retrieve_relevant_chunks(question,top_k=3):
    question_embedding=embedding_model.encode(question).tolist()
    results=collection.query(
    query_embeddings=[question_embedding],
     n_results=top_k,
    )
    return results


In [13]:
test_question='what is the etl and how does it work in data engineering?'

print('test question:', test_question)

results = retrieve_relevant_chunks(test_question, top_k=3)

for i, (doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):

    print(f'\nResult {i+1}')

    print('subject:', meta['subject'])
    print('topic:', meta['topic'])
    print('distance:', dist)
    print('content:', doc[:100])

test question: what is the etl and how does it work in data engineering?

Result 1
subject: Data Engineering
topic: ETL Pipelines
distance: 0.2517554461956024
content: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc

Result 2
subject: Data Engineering
topic: APIs and Data Collection
distance: 1.07559072971344
content: An API or Application Programming Interface allows two software applications to talk to each other. 

Result 3
subject: Data Engineering
topic: SQL Databases
distance: 1.3636817932128906
content: A database is an organized collection of data stored electronically. SQL or Structured Query Languag


In [14]:
test_question='what is generative ai and how does it work?'

print('test question:', test_question)

results = retrieve_relevant_chunks(test_question, top_k=3)

for i, (doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):

    print(f'\nResult {i+1}')

    print('subject:', meta['subject'])
    print('topic:', meta['topic'])
    print('distance:', dist)
    print('content:', doc[:100])

test question: what is generative ai and how does it work?

Result 1
subject: Generative AI
topic: Retrieval Augmented Generation
distance: 1.0418833494186401
content: RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant docu

Result 2
subject: Generative AI
topic: Large Language Models
distance: 1.2242777347564697
content: A Large Language Model or LLM is an AI model trained on massive amounts of text data. It can generat

Result 3
subject: Generative AI
topic: Prompt Engineering
distance: 1.2307566404342651
content: Prompt engineering is the practice of designing effective instructions or inputs for AI language mod


In [19]:
def build_context_from_results(results):
    context_parts=[]

    for i, (doc, meta) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0]
    )):

        chunk_text=f"""
[source:{i+1}: {meta['subject']} - {meta['topic']}]

{doc}
"""

        context_parts.append(chunk_text)

    context_str='\n\n-----\n\n'.join(context_parts)

    return context_str


context=build_context_from_results(results)

print(context)


[source:1: Generative AI - Retrieval Augmented Generation]

RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to answer questions about specific data.


-----


[source:2: Generative AI - Large Language Models]

A Large Language Model or LLM is an AI model trained on massive amounts of text data. It can generate human-like text answer questions summarize documents and perform many language tasks. Examples include GPT Claude and LLaMA.


-----


[source:3: Generative AI - Prompt Engineering]

Prompt engineering is the practice of designing effective instructions or inputs for AI language models. A good prompt gives the model clear context specific instructions and examples if needed. Techniques include zero-shot few-shot role-based and chain-of-thought prompting.



In [20]:
def generate_rag_answer(question, context):
    system_prompt = """
You are a helpful academic assistant for engineering students.

You will be given context retrieved from a college knowledge base, and a student's question.

RULES:
1. Answer ONLY using the information provided in the context below.
2. If the answer is not found in the context, say exactly:
   "I don't have enough information in my knowledge base to answer this question."
3. Do not use your general training knowledge.
4. Keep answers clear, accurate, and beginner-friendly.
5. Mention which source the information came from when possible.
"""

    user_prompt = f"""Context from Knowledge Base:

{context}

---

Student's Question: {question}

Please answer the question based only on the context provided above."""

    response = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )

    answer = response.choices[0].message.content
    return answer


print('rag generation function defined')

rag generation function defined


In [23]:
def ask_college_assistant(question, top_k=3, verbose=True):

    if verbose:
        print('question:', question)
        print('---')

    results = retrieve_relevant_chunks(question, top_k=top_k)

    if verbose:
        print('retrieved:', top_k, 'chunks from the knowledge base')

        for i, meta in enumerate(results['metadatas'][0]):
            print(f"{i+1}. {meta['subject']} - {meta['topic']}")

        print('---')

    context = build_context_from_results(results)

    if verbose:
        print('context built:', len(context))

    answer = generate_rag_answer(question, context)

    return answer

In [24]:
GROP_API_KEY='--------------------------------------------------'
os.environ['GROQ_API_KEY']=GROP_API_KEY
groq_client=Groq(api_key=GROP_API_KEY)
print('groq is initialised')

groq is initialised


In [28]:
question_1='what is the etl and what are its three main stages?'
answer_1=ask_college_assistant(question_1,top_k=3,verbose=True)
print('\nFINAL ANSWER:\n')
print(answer_1)


question: what is the etl and what are its three main stages?
---
retrieved: 3 chunks from the knowledge base
1. Data Engineering - ETL Pipelines
2. Generative AI - Prompt Engineering
3. Generative AI - Retrieval Augmented Generation
---
context built: 943

FINAL ANSWER:

According to the context from [source:1: Data Engineering - ETL Pipelines], ETL stands for Extract Transform Load. 

The three main stages of ETL are:

1. Extract: This stage involves collecting raw data from different sources.
2. Transform: This stage involves transforming the raw data into a clean and structured format.
3. Load: This stage involves loading the transformed data into a database or data warehouse for analysis.

Source: [source:1: Data Engineering - ETL Pipelines]


In [29]:
question_2='what is generative ai and how does it work?'

answer_2 = ask_college_assistant(question_2,top_k=3,verbose=True)

print('\nFINAL ANSWER:\n')
print(answer_2)

question: what is generative ai and how does it work?
---
retrieved: 3 chunks from the knowledge base
1. Generative AI - Retrieval Augmented Generation
2. Generative AI - Large Language Models
3. Generative AI - Prompt Engineering
---
context built: 958

FINAL ANSWER:

Based on the context provided, I can answer the student's question.

Generative AI refers to a type of AI model that can generate human-like text, answer questions, summarize documents, and perform many language tasks. This is mentioned in [source:2: Generative AI - Large Language Models].

As for how it works, Generative AI uses a technique called Retrieval Augmented Generation (RAG) [source:1: Generative AI - Retrieval Augmented Generation]. In RAG, the AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to answer questions about specific data.

Additionally, the effectiveness of Generative AI also de

In [30]:
question_3='what is machine learning and what are its types?'
answer_3 = ask_college_assistant(question_3,top_k=3,verbose=True)
print('\nFINAL ANSWER:\n')
print(answer_3)

question: what is machine learning and what are its types?
---
retrieved: 3 chunks from the knowledge base
1. Machine Learning - Supervised Learning
2. Machine Learning - Decision Trees
3. Machine Learning - Random Forest
---
context built: 889

FINAL ANSWER:

Machine learning is a type of learning where the model learns from labeled data. It is given input features and correct output labels and it learns to predict outputs for new unseen inputs. 

There are at least two types of machine learning mentioned in the context:

1. Supervised learning: This type of machine learning is where the model learns from labeled data. It includes examples such as classification and regression. [source:1: Machine Learning - Supervised Learning]

2. Ensemble learning (specifically, Random Forest): This type of machine learning builds multiple decision trees and combines their predictions. It is more accurate and robust than a single decision tree. [source:3: Machine Learning - Random Forest]

Additiona

In [31]:
question_4='who is the captain of indian cricket team?'
answer_4 = ask_college_assistant(question_4,top_k=3,verbose=True)
print('\nFINAL ANSWER:\n')
print(answer_4)

question: who is the captain of indian cricket team?
---
retrieved: 3 chunks from the knowledge base
1. Machine Learning - Random Forest
2. Generative AI - Large Language Models
3. Machine Learning - Supervised Learning
---
context built: 892

FINAL ANSWER:

I don't have enough information in my knowledge base to answer this question.
